In [1]:
from pathlib import Path
from zipfile import ZipFile
import requests

DATA_URL = (
    "https://archive.ics.uci.edu/static/public/235/"
    "individual+household+electric+power+consumption.zip"
)

RAW_DIR = Path("data/raw")
ZIP_PATH = RAW_DIR / "household_power_consumption.zip"
DATA_PATH = RAW_DIR / "household_power_consumption.txt"

RAW_DIR.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    print("Downloading dataset...")

    response = requests.get(DATA_URL, timeout=120)
    response.raise_for_status()
    ZIP_PATH.write_bytes(response.content)

    print(f"Downloaded: {ZIP_PATH}")
else:
    print("ZIP file already exists. Download skipped.")

if not DATA_PATH.exists():
    print("Extracting dataset...")

    with ZipFile(ZIP_PATH, "r") as zip_file:
        zip_file.extractall(RAW_DIR)

    print(f"Extracted: {DATA_PATH}")
else:
    print("Dataset already extracted.")

print(f"Dataset ready: {DATA_PATH.resolve()}")

Downloaded: data\raw\household_power_consumption.zip
Extracting dataset...
Extracted: data\raw\household_power_consumption.txt
Dataset ready: E:\My_Projects\AIM\capstone\data\raw\household_power_consumption.txt


In [ ]:
import pandas as pd

# Load the dataset into a pandas DataFrame
df = pd.read_csv(
    DATA_PATH,
    sep=";",
    na_values="?",
    low_memory=False
)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
df.head()

Rows: 2,075,259
Columns: 9


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


In [3]:
# Create the timestamp

df["datetime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    format="%d/%m/%Y %H:%M:%S"
)

df = (
    df.drop(columns=["Date", "Time"])
      .set_index("datetime")
      .sort_index()
)

df.head()

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
datetime,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


In [4]:
# Convert measurements to numeric values
measurement_columns = df.columns

df[measurement_columns] = df[measurement_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2075259 entries, 2006-12-16 17:24:00 to 2010-11-26 21:02:00
Data columns (total 7 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Global_active_power    float64
 1   Global_reactive_power  float64
 2   Voltage                float64
 3   Global_intensity       float64
 4   Sub_metering_1         float64
 5   Sub_metering_2         float64
 6   Sub_metering_3         float64
dtypes: float64(7)
memory usage: 126.7 MB
